## Installation and Verification of the Module

### Installing llama-cpp-python

For CPU-only inference:
`uv add llama-cpp-python`

For NVIDIA GPU support:
`CMAKE_ARGS="-DGGML_CUDA=ON -DGGML_NATIVE=OFF" uv add llama-cpp-python`

### Commands for Installing HuggingFace Hub

Installing huggingface-hub:
`uv add huggingface-hub`


In [1]:
# Verify GPU offloading support

from llama_cpp import llama_supports_gpu_offload
print(llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 3770 MiB):
  Device 0: NVIDIA GeForce RTX 2050, compute capability 8.6, VMM: yes, VRAM: 3770 MiB


True


## Model Inference Example

In [2]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


# Download the model file from Hugging Face
model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-0.5B-Instruct-GGUF",
    filename="Qwen2.5-0.5B-Instruct-Q4_K_M.gguf",
    local_dir="./models",
)

# Initialize the model for inference using the downloaded file
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=True,
)

/home/vishwjeet/Desktop/Projects/LinguSync/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_model_loader: loaded meta data with 38 key-value pairs and 290 tensors from /home/vishwjeet/Desktop/Projects/LinguSync/inference_engine/models/Qwen2.5-0.5B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 0.5B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruc

In [3]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": "tell me about LLM"
        }
    ],
    max_tokens=100,
)

print(response["choices"][0]["message"]["content"])

CUDA Graph id 33 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA Graph id 33 reused
CUDA

LLM stands for Large Language Model, which is a type of artificial intelligence that can generate human-like text. These models are designed to understand and generate human language, and they can be used for a wide range of tasks such as language translation, text summarization, and generating creative content.

LLMs are different from traditional machine learning models, which are designed to learn from data and make predictions or classifications. LLMs are specifically designed to learn and generate text, and they can be trained on large


In [4]:
# Cleanup

import gc
del llm
gc.collect()

~llama_context:      CUDA0 compute buffer size is 298.5000 MiB, matches expectation of 298.5000 MiB
~llama_context:  CUDA_Host compute buffer size is   8.0098 MiB, matches expectation of   8.0098 MiB


82

## Downloading Finilize Models

In [19]:
from huggingface_hub import hf_hub_download

repo = "ggml-org/Qwen3-ASR-0.6B-GGUF"

model_path = hf_hub_download(
    repo_id=repo,
    filename="Qwen3-ASR-0.6B-Q8_0.gguf",
    local_dir="./models",
)

mmproj_path = hf_hub_download(
    repo_id=repo,
    filename="mmproj-Qwen3-ASR-0.6B-bf16.gguf",
    local_dir="./models",
)

In [14]:
from huggingface_hub import snapshot_download

repo = "Qwen/Qwen3-ForcedAligner-0.6B"

model_path = snapshot_download(
    repo_id=repo,
    local_dir="./models/Qwen3-ForcedAligner-0.6B",
)

print(model_path)

/home/vishwjeet/Desktop/Projects/LinguSync/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 10 files: 100%|██████████| 10/10 [07:16<00:00, 43.65s/it] 

/home/vishwjeet/Desktop/Projects/LinguSync/inference_engine/models/Qwen3-ForcedAligner-0.6B


In [4]:
from pathlib import Path
import subprocess
import shutil

base_path = "/home/vishwjeet/Desktop/Projects/LinguSync"

def check_llama_cpp(base_path):
    llama_dir = Path(base_path) / "llamacpp"
    llama_server = llama_dir / "bin" / "llama-server"

    # Not installed
    if not llama_server.is_file():
        return False, llama_dir

    # Installed, but verify it actually works
    try:
        result = subprocess.run(
            [str(llama_server), "--version"],
            capture_output=True,
            text=True,
            timeout=10
        )

        if result.returncode == 0:
            print("llama.cpp is installed and working.")
            return True, llama_dir

    except (OSError, subprocess.SubprocessError):
        pass

    # Installation is broken
    print("llama.cpp installation is broken.")
    print(f"Removing: {llama_dir}")

    shutil.rmtree(llama_dir, ignore_errors=True)

    return False, llama_dir

In [6]:
installed, llama_dir = check_llama_cpp(base_path)

if not installed:
    print("llama.cpp is missing. Starting installer...")

    # Run your installer
    subprocess.run(
        ["./install_llamacpp.sh", base_path],
        check=True
    )

llama.cpp is missing. Starting installer...
 llama.cpp CUDA Installer

Base path : /home/vishwjeet/Desktop/Projects/LinguSync
Install   : /home/vishwjeet/Desktop/Projects/LinguSync/llamacpp

[1/5] Checking NVIDIA GPU...
[OK] NVIDIA GPU: NVIDIA GeForce RTX 2050

[2/5] Checking CUDA Toolkit...
[OK] CUDA Toolkit: 12.0

[3/5] Checking build tools...
[OK] git
[OK] cmake
[OK] g++

[4/5] Checking existing llama.cpp installation...
[INFO] llama.cpp is not installed.

[5/5] Downloading and building llama.cpp...
[INFO] Temporary directory:
       /tmp/tmp.em0SuQZpsM

[INFO] Cloning llama.cpp...


Cloning into '/tmp/tmp.em0SuQZpsM/llama.cpp'...


[INFO] Configuring CUDA build...
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.0-dev
-- Found Git: /usr/bin/git (found version "2.43.0") 
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc


CMAKE_BUILD_TYPE=Release


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Found CUDAToolkit: /usr/include (found version "12.0.140") 
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 12.0.140
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Using CMAKE_CUDA_ARCHITECTURES=86-real CMAKE_CUDA_ARCHITECTURES_NATIVE=86-real
-- FlashAttent

In static member function ‘static _Tp* std::__copy_move<_IsMove, true, std::random_access_iterator_tag>::__copy_m(const _Tp*, const _Tp*, _Tp*) [with _Tp = ggml_op; bool _IsMove = false]’,
    inlined from ‘_OI std::__copy_move_a2(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:494:141,
    inlined from ‘_OI std::__copy_move_a1(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:522:40,
    inlined from ‘_OI std::__copy_move_a(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:529:25,
    inlined from ‘_OI std::copy(_II, _II, _OI) [with _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:619:64,
    inlined from ‘static _ForwardIterator std::__uninitialized_copy<true>::__uninit_copy(_InputIterator, _InputIterator, _ForwardIterator) [wit

[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/norm.cu.o
[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/opt-step-adamw.cu.o
[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/opt-step-sgd.cu.o
[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/out-prod.cu.o
[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad.cu.o
[ 14%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad_reflect_1d.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool1d.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool2d.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/quantize.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/roll.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/rope.cu.o
[ 15%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/g

In [ ]:
import os
import subprocess

llama_dir = "/home/vishwjeet/Desktop/Projects/LinguSync/llamacpp"

llama_server = f"{llama_dir}/bin/llama-server"

model = "/home/vishwjeet/Desktop/Projects/LinguSync/inference_engine/models/Qwen3-ASR-0.6B-Q8_0.gguf"

mmproj = "/home/vishwjeet/Desktop/Projects/LinguSync/inference_engine/models/mmproj-Qwen3-ASR-0.6B-bf16.gguf"

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{llama_dir}/lib:" + env.get("LD_LIBRARY_PATH", "")

process = subprocess.Popen(
    [
        llama_server,
        "-m", model,
        "--mmproj", mmproj,
        "-ngl", "99",
        "--port", "8080",
    ],
    env=env
)

0.00.118.735 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.119.354 W srv  llama_server: -----------------
0.00.119.358 W srv  llama_server: CORS is set to allow all origins ('*') and no API key is set
0.00.119.358 W srv  llama_server: this can be a security risk (cross-origin attacks)
0.00.119.358 W srv  llama_server: more info: https://github.com/ggml-org/llama.cpp/pull/25655
0.00.119.359 W srv  llama_server: -----------------
0.00.120.819 I srv    load_model: loading model '/home/vishwjeet/Desktop/Projects/LinguSync/inference_engine/models/Qwen3-ASR-0.6B-Q8_0.gguf'
0.00.939.438 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
0.01.953.029 I cmn          init: llama threadpool init, n_threads = 4
0.02.473.288 W init_audio: audio input is in experimental stage and may have reduced quality:
    https://github.com/ggml-org/llama.cpp/discussions/13759
0.

In [13]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="none",
)

input_audio = "/home/vishwjeet/Desktop/Projects/LinguSync/RefVideo/output.mp3"

with open(input_audio, "rb") as f:
    result = client.audio.transcriptions.create(
        model="Qwen3-ASR-0.6B",
        file=f,
        response_format="json",
    )

print(result)

11.46.490.307 I slot get_availabl: id  1 | task -1 | selected slot by LRU, t_last = -1
11.46.512.359 I slot launch_slot_: id  1 | task 166 | processing task, is_child = 0
11.46.537.436 W slot   operator(): id  1 | task 166 | need to evaluate at least 1 token for each active slot (n_past = 288, task.n_tokens() = 288)
11.46.537.439 W slot   operator(): id  1 | task 166 | n_past was set to 287


Transcription(text="language English<asr_text>Okay, sweetie, do that little dance again. Okay, post. Oh, people are gonna love that. I could watch this all day. Who are you? Oh, uh, Todd underscore nineteen seventy five. What are you doing here? You just made another post of my favorite little girl. She's almost six now, huh? Um, yeah, she's looking.", languages=None, logprobs=None, usage=UsageTokens(input_tokens=288, output_tokens=81, total_tokens=369, type='tokens', input_token_details=None, input_tokens_details={'cached_tokens': 287}), type='transcript.text.done')


11.47.215.623 I slot print_timing: id  1 | task 166 | prompt eval time =      10.99 ms /     1 tokens (   10.99 ms per token,    90.98 tokens per second)
11.47.215.626 I slot print_timing: id  1 | task 166 |        eval time =     667.19 ms /    81 tokens (    8.34 ms per token,   119.91 tokens per second)
11.47.215.627 I slot print_timing: id  1 | task 166 |       total time =     678.18 ms /    82 tokens
11.47.215.628 I slot print_timing: id  1 | task 166 |    graphs reused =        241
11.47.215.644 I slot      release: id  1 | task 166 | stop processing: n_tokens = 368, truncated = 0
